In [6]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os
load_dotenv()

def load_model():
    api_key = os.getenv("GOOGLE_API_KEY1")
    model_client = OpenAIChatCompletionClient(model="gemini-1.5-flash", api_key=api_key)
    return model_client
model_client = load_model() 

ModuleNotFoundError: No module named 'openai'

In [7]:
from autogen_agentchat.agents import AssistantAgent

def load_agents():
    model_client = load_model()
    agent1 = AssistantAgent(
        name="agent1",
        model_client=model_client,
        system_message=""
    )
    return agent1


In [8]:
pip install autogen_ext

Note: you may need to restart the kernel to use updated packages.


In [9]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

model_client = OllamaChatCompletionClient(model="mistral")

In [10]:
from autogen_agentchat.agents import AssistantAgent

def load_agents_ollama():
    model_client = OllamaChatCompletionClient(model="mistral")
    agent1 = AssistantAgent(
        name="agent1",
        model_client=model_client,
        system_message=""
    )
    return agent1

In [11]:
agent = load_agents_ollama()
await agent.run(task="hello")

TaskResult(messages=[TextMessage(id='abebf1df-e69b-4b4f-aac2-0ffd6211f666', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 16, 10, 30, 16, 394055, tzinfo=datetime.timezone.utc), content='hello', type='TextMessage'), TextMessage(id='c7721386-3f67-4934-a588-e846bb53d147', source='agent1', models_usage=RequestUsage(prompt_tokens=6, completion_tokens=34), metadata={}, created_at=datetime.datetime(2025, 7, 16, 10, 30, 26, 560835, tzinfo=datetime.timezone.utc), content=" Hello there! How can I help you today? If you have any questions or need assistance with something, feel free to ask. I'm here to help.", type='TextMessage')], stop_reason=None)

In [12]:
# agent = load_agents()
# await agent.run(task="hello")

In [13]:
from autogen_core.tools import FunctionTool
from typing import Any

def create_agent(name: str="", model_client: Any="", reflect_on_tool_use: bool=True, tools: list=[]) -> str:
    """
    Tool for creating Autogen agent.

    Args:
        name (str): The name of the agent.
        model_client (OpenAIChatCompletionClient, optional): The model client to use for the agent. Defaults to model_client.
        reflect_on_tool_use (bool, optional): Whether the agent should reflect on tool use. Defaults to True.
        tools: Tools that should be binded with model client or agent

    Returns:
        str: A string representation of the agent initialization code.
    """
    system_message = ""
    tools = []
    agent_string = f"""
    agent = AssistantAgent(
        name={name},
        model_client={model_client},
        system_message={system_message},
        tools={tools},
        reflect_on_tool_use={reflect_on_tool_use},
    )
    """
    return agent_string
create_agent_tool = FunctionTool(create_agent, description="A tool for creating agent by passing necessary arguments")

In [14]:
agent_tools = [create_agent_tool] 
meta_agent = AssistantAgent(
        name="Autogen_Meta_Agent",
        model_client=model_client,
        system_message="""
        You are any Autogen Meta Agent tasked with creating autogen agents using tool {agent_tools}
        you have create agents according to user request.
        frist you have to plan , how to make this agent.
        then , when you have to create agent , just use your tool {agent_tools} by passing necessary arguments.
        """,
        tools=agent_tools,
        reflect_on_tool_use=True,
    )


In [ ]:
prompt_agent = AssistantAgent(
    name="prompt_writing_agent",
    model_client=model_client,
    system_message="""
    You are an prompt writing agent tasked with writing system prompt to agents according to their target or task.
    """
)

In [ ]:
tool_agent = AssistantAgent(
    name="tool_making_agent",
    model_client=model_client,
    system_message="""
    You are an tool making agent tasked with making approprate tools for agents to use .
    You have to give it like:
    def tool_name:
        <tool code>

    tool_name = FunctionTool(tool_name, description="<appropriate description>")

    because you are an tool making agent as a part full autogen agent making agent.
    """
)

In [ ]:
planning_agent = AssistantAgent(
    name="Planning_agent",
    model_client=model_client,
    system_message="""
    You are an agent workflow planning agent. 
    you are a part of autogen full agent code writing agent .
    So, you have to carefully plan how the agent workflow should be according user request.
    """
)

In [15]:
async for message in meta_agent.run_stream(task="create a simple agent for summarize texts"):
    print(message)

id='83e4f9b9-fb4f-42fb-a4c6-e10ac6723be3' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 16, 10, 30, 26, 616709, tzinfo=datetime.timezone.utc) content='create a simple agent for summarize texts' type='TextMessage'
id='faa522e7-fc99-4f25-82a7-6e6b5666dbb5' source='Autogen_Meta_Agent' models_usage=RequestUsage(prompt_tokens=200, completion_tokens=128) metadata={} created_at=datetime.datetime(2025, 7, 16, 10, 31, 15, 751556, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='0', arguments='{"model_client": "SummarizationModelClient", "name": "TextSummaryAgent", "reflect_on_tool_use": true, "tools": ["SummarizationTool"]}', name='create_agent')] type='ToolCallRequestEvent'
id='c22b84b9-ae33-4312-9add-c7fea6b115ac' source='Autogen_Meta_Agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 16, 10, 31, 15, 755654, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='\n    agent = AssistantAgent(\n        name=Text

In [ ]:
[id='83e4f9b9-fb4f-42fb-a4c6-e10ac6723be3' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 16, 10, 30, 26, 616709, tzinfo=datetime.timezone.utc) content='create a simple agent for summarize texts' type='TextMessage'
,id='faa522e7-fc99-4f25-82a7-6e6b5666dbb5' source='Autogen_Meta_Agent' models_usage=RequestUsage(prompt_tokens=200, completion_tokens=128) metadata={} created_at=datetime.datetime(2025, 7, 16, 10, 31, 15, 751556, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='0', arguments='{"model_client": "SummarizationModelClient", "name": "TextSummaryAgent", "reflect_on_tool_use": true, "tools": ["SummarizationTool"]}', name='create_agent')] type='ToolCallRequestEvent'
,id='c22b84b9-ae33-4312-9add-c7fea6b115ac' source='Autogen_Meta_Agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 16, 10, 31, 15, 755654, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='\n    agent = AssistantAgent(\n        name=TextSummaryAgent,\n        model_client=SummarizationModelClient,\n        system_message=,\n        tools=[],\n        reflect_on_tool_use=True,\n    )\n    ', name='create_agent', call_id='0', is_error=False)] type='ToolCallExecutionEvent'
,id='5b2bc386-105c-4526-8c6c-ab5b7ce990b6' source='Autogen_Meta_Agent' models_usage=RequestUsage(prompt_tokens=129, completion_tokens=298) metadata={} created_at=datetime.datetime(2025, 7, 16, 10, 32, 16, 15526, tzinfo=datetime.timezone.utc) content=' That\'s a good start for creating an agent named `TextSummaryAgent` that uses the `SummarizationModelClient`. However, you will need to initialize the `SummarizationModelClient` and also provide a system message for the agent. The system message is a prompt used by the assistant to guide its responses and can be thought of as the assistant\'s starting point when generating responses.\n\nHere is an updated version with the initialization of the `SummarizationModelClient` and a simple system message:\n\n```python\nfrom langchain.agents import AssistantAgent\nfrom langchain.llms import SummarizationModelLLM\n\nclass TextSummaryAgent(AssistantAgent):\n    def __init__(self):\n        self.model = SummarizationModelLLM()\n        super().__init__(\n            name=TextSummaryAgent,\n            model_client=self.model,\n            system_message="You are a text summarizer. Given a long piece of text, you should generate a concise summary of the main points.",\n            tools=[],\n            reflect_on_tool_use=True,\n        )\n```\n\nIn this example, I\'ve used `SummarizationModelLLM` which is an interface for language models that can perform summarization. You would replace it with your preferred language model or create a custom one based on your requirements.' type='TextMessage'
,messages=[TextMessage(id='83e4f9b9-fb4f-42fb-a4c6-e10ac6723be3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 16, 10, 30, 26, 616709, tzinfo=datetime.timezone.utc), content='create a simple agent for summarize texts', type='TextMessage'), ToolCallRequestEvent(id='faa522e7-fc99-4f25-82a7-6e6b5666dbb5', source='Autogen_Meta_Agent', models_usage=RequestUsage(prompt_tokens=200, completion_tokens=128), metadata={}, created_at=datetime.datetime(2025, 7, 16, 10, 31, 15, 751556, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='0', arguments='{"model_client": "SummarizationModelClient", "name": "TextSummaryAgent", "reflect_on_tool_use": true, "tools": ["SummarizationTool"]}', name='create_agent')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='c22b84b9-ae33-4312-9add-c7fea6b115ac', source='Autogen_Meta_Agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 16, 10, 31, 15, 755654, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='\n    agent = AssistantAgent(\n        name=TextSummaryAgent,\n        model_client=SummarizationModelClient,\n        system_message=,\n        tools=[],\n        reflect_on_tool_use=True,\n    )\n    ', name='create_agent', call_id='0', is_error=False)], type='ToolCallExecutionEvent'), TextMessage(id='5b2bc386-105c-4526-8c6c-ab5b7ce990b6', source='Autogen_Meta_Agent', models_usage=RequestUsage(prompt_tokens=129, completion_tokens=298), metadata={}, created_at=datetime.datetime(2025, 7, 16, 10, 32, 16, 15526, tzinfo=datetime.timezone.utc), content=' That\'s a good start for creating an agent named `TextSummaryAgent` that uses the `SummarizationModelClient`. However, you will need to initialize the `SummarizationModelClient` and also provide a system message for the agent. The system message is a prompt used by the assistant to guide its responses and can be thought of as the assistant\'s starting point when generating responses.\n\nHere is an updated version with the initialization of the `SummarizationModelClient` and a simple system message:\n\n```python\nfrom langchain.agents import AssistantAgent\nfrom langchain.llms import SummarizationModelLLM\n\nclass TextSummaryAgent(AssistantAgent):\n    def __init__(self):\n        self.model = SummarizationModelLLM()\n        super().__init__(\n            name=TextSummaryAgent,\n            model_client=self.model,\n            system_message="You are a text summarizer. Given a long piece of text, you should generate a concise summary of the main points.",\n            tools=[],\n            reflect_on_tool_use=True,\n        )\n```\n\nIn this example, I\'ve used `SummarizationModelLLM` which is an interface for language models that can perform summarization. You would replace it with your preferred language model or create a custom one based on your requirements.', type='TextMessage')] stop_reason=None
]

SyntaxError: invalid syntax (3258724533.py, line 15)

In [17]:
response = await meta_agent.run(task="create a simple agent for summarize texts")

In [30]:
import pprint
for i in response:
    result = list(i)
    break

In [37]:
pprint.pprint(result[1][1].content)

(' To create a simple agent for summarizing text using the provided '
 '`create_agent` function, we can modify the existing code as follows:\n'
 '\n'
 '```python\n'
 '[{"name": "create_agent", "arguments": '
 '{"model_client":"SummarizationModelClient","name":"TextSummaryAgent","reflect_on_tool_use":True,"tools":["SummarizationTool"]}}]\n'
 '```\n'
 '\n'
 'In this example, the `create_agent` function is used to create an agent '
 'named `TextSummaryAgent`, which utilizes the `SummarizationModelClient`. The '
 'summarization tool will be a built-in one in this case.')
